# AF de MEGADADOS

## Permissões de uso

Proibido publicar ou compartilhar este material com terceiros.

## Iniciar

A prova tem duração de **2h30**. Veja mais informações no Blackboard!

Não se esqueça de **entregar** o notebook ao final da prova. Deixe as células **executadas**!

## Identifique-se

**NOME**: Pedro Pereira Cecílio Ventura

## Insper autograding!

Para receber feedback dos exercício, iremos utilizar o `insper autograding`.

**Importante**: você precisará do `.env`, utilize o mesmo procedimento feito na aula de **Spark**!

In [1]:
# !pip install -U git+https://github.com/macielcalebe/insperautograding.git

## Como resolver os exercícios?

No exercício 1, rode o Docker para resolver as questões de programação funcional.

Nas demais, crie a base da prova em sua máquina. Utilize o notebook, MySQL Workbench ou o conector para testar as queries e soluções. Quando estiver bastante certo de que a resposta está correta, faça a submissão para o servidor.

Para macOS e linux, utilize:

```bash
docker run \
    -it \
    --rm \
    -p 8888:8888 \
    -p 4040:4040 \
    -v "`pwd`":/home/jovyan/work \
    jupyter/pyspark-notebook


```

Se estiver no Windows estes comandos, utilize:

- No Powershell: `docker run -it --rm -p 8888:8888 -p 4040:4040 -v ${PWD}:/home/jovyan/work jupyter/pyspark-notebook`

- No Prompt de comando: `docker run -it --rm -p 8888:8888 -p 4040:4040 -v %cd%:/home/jovyan/work jupyter/pyspark-notebook`

Agora abra esse notebook lá no container!

## Import das bibliotecas

Vamos realizar o import das bibliotecas.

In [2]:
import os
import insperautograder.jupyter as ia
from functools import partial
from pyspark.sql import SparkSession
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [3]:
spark = SparkSession.builder.appName("MinhaAplicacao").getOrCreate()
sc = spark.sparkContext

24/11/25 07:37:47 WARN Utils: Your hostname, MacBook-Pro-de-Pedro.local resolves to a loopback address: 127.0.0.1; using 10.102.0.69 instead (on interface en0)
24/11/25 07:37:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/25 07:37:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Notas

As primeiras quatro questões, que possuem correção automática, valem 75% da nota da prova.

Na API de correção automática a nota de cada questão será ponderada pelo seu peso. A nota será apresentada no intervalo 0 a 10, multiplique por 0.75 para saber a nota final considerando toda a prova.

Para conferir a nota da correção automática da prova, utilize:

In [129]:
ia.grades(task="af_md_24_2")

|    | Atividade   | Exercício   |   Peso |   Nota |
|---:|:------------|:------------|-------:|-------:|
|  0 | af_md_24_2  | ex01a       |      2 |     10 |
|  1 | af_md_24_2  | ex01b       |      4 |     10 |
|  2 | af_md_24_2  | ex01c       |      5 |     10 |
|  3 | af_md_24_2  | ex02        |      2 |      0 |
|  4 | af_md_24_2  | ex03        |      4 |     10 |
|  5 | af_md_24_2  | ex04        |      4 |     10 |

In [130]:
ia.grades(by="TASK", task="af_md_24_2")

|    | Tarefa     |   Nota |
|---:|:-----------|-------:|
|  0 | af_md_24_2 |   9.05 |

**Exercício 1 - Monitoramento de Expedição ao Ártico**

<img src="img/arctic-scientific-research-stockcake.jpg">

Durante expedições ao Ártico, equipes de exploradores monitoram atividades climáticas e registram relatórios sobre as condições encontradas durante saídas da base.

Cada explorador, ao retornar à base, faz o upload dos relatórios. Os relatórios são registrados em um formato de log de texto, com "`|`" como separador entre as colunas e cada linha representando um relato.

O formato de cada linha de log é:

```python
"<ID_RELATO>|<NIVEL_RISCO>|<COD_ESTACAO>|<DIA>-<MES>-<ANO>|<LATITUDE>|<LONGITUDE>|<EXPLORADOR>|<DADOS>"
```

Onde:

- `<ID_RELATO>`: um número inteiro que identifica o relato.
- `<NIVEL_RISCO>`: string com o nível de risco observado (ex: `severo, moderado, leve`).
- `<COD_ESTACAO>`: número inteiro que identifica a estação base de onde o relato foi enviado.
- `<DIA>`: número inteiro representando o dia.
- `<MES>`: string que representa o mês.
- `<ANO>`: número inteiro representando o ano.
- `<LATITUDE>` e `<LONGITUDE>`: coordenadas geográficas do local do evento.
- `<EXPLORADOR>`: nome do explorador que produziu o relato.
- `<DADOS>`: string com a descrição do que foi observado ou as condições de risco enfrentadas.

### Exemplo de Dados do Log

Aqui está um exemplo de algumas linhas de log para ilustrar o formato com geolocalização e `|` como separador:

```python
"00124|severo|12|15-Fevereiro-2024|78.2234|-103.4567|Adams|Ventos extremos, visibilidade quase nula."
"00125|moderado|12|16-Fevereiro-2024|77.8901|-102.3456|Clark|Temperaturas abaixo de -40ºC, movimentação de placas de gelo."
"00126|leve|15|18-Março-2024|79.0123|-101.2345|Wilson|Observação de vida selvagem rara na região noroeste."
"00127|severo|15|20-Março-2024|78.9876|-100.1234|Adams|Rachaduras no solo glacial, risco de fissuras."
"00128|leve|12|25-Abril-2024|78.3456|-102.5678|Clark|Estudo das formações de gelo e registro de amostras."
```

**a)** Crie uma função `total_logs_por_risco` que recebe (nesta ordem):

- o RDD no formato acima.
- uma string com o nível de risco.

E retorna a quantidade total de logs com este nível de risco.

In [22]:
def total_logs_por_risco(rdd, risco):
    splitado = rdd.map(lambda x: x.split("|")[1])
    filtrado = splitado.filter(lambda x: str(x) == str(risco))
    return filtrado.count()

rdd = sc.textFile("data/log_artico1.txt")

print(total_logs_por_risco(rdd, "leve"))

4


Um teste local:

In [23]:
# Garantir uso do RDD correto
rdd = sc.textFile("data/log_artico1.txt")

assert total_logs_por_risco(rdd, "leve") == 4
assert total_logs_por_risco(rdd, "moderado") == 16
assert total_logs_por_risco(rdd, "inexistente") == 0

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [24]:
ia.sender(answer="total_logs_por_risco", task="af_md_24_2", question="ex01a", answer_type="pycode")

interactive(children=(Button(description='Enviar ex01a', style=ButtonStyle()), Output()), _dom_classes=('widge…

**b)** Crie uma função `top_k_meses(rdd, k)` que recebe um RDD no formato descrito e um inteiro `k`.

A função deve retornar uma lista com os `k` meses com mais relatos (logs), seguido de suas quantidades de relatos, ordenados de forma decrescente pelo número de relatos.

In [30]:
def top_k_meses(rdd, k):
    datas = rdd.map(lambda x: x.split("|")[3])
    meses = datas.map(lambda x: (x.split("-")[1], 1))
    qtd_meses = meses.reduceByKey(lambda x, y: x+y)
    qm_ordenado = qtd_meses.sortBy(lambda x: x[1], ascending=False)
    return qm_ordenado.take(k)

rdd = sc.textFile("data/log_artico1.txt")

print(top_k_meses(rdd, 2))

[('Agosto', 14), ('Julho', 10)]


Alguns testes locais:

In [31]:
# Garantir uso do RDD correto
rdd = sc.textFile("data/log_artico1.txt")

assert top_k_meses(rdd, 2) == [('Agosto', 14), ('Julho', 10)]
assert top_k_meses(rdd, 3) == [('Agosto', 14), ('Julho', 10), ('Novembro', 9)]

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [32]:
ia.sender(answer="top_k_meses", task="af_md_24_2", question="ex01b", answer_type="pycode")

interactive(children=(Button(description='Enviar ex01b', style=ButtonStyle()), Output()), _dom_classes=('widge…

**c)** Crie uma função `frequencia_palavras(rdd, risco, k)` onde o RDD recebido está no formato do enunciado, `risco` é uma string relativa ao nível de risco do relato e `k` é um inteiro.

Considerando apenas os relatos com nível de risco informado, sua função deve retornar as `k` palavras mais frequentes, além de suas frequências relativas (porcentagem) no texto das **mensagens** contidas nos relatórios (`<DADOS>`).

**Dicas**:

- Transforme o texto para minúsculo.
- Limpo pontos e vírgulas (`.`, `,`).
- Na mensagem considerada, não devem haver espaços antes ou após a mensagem do relatório.
- Considere porcentagem no intervalo zero a cem.
- Considere a fórmula de porcentagem `x / n * 100`, onde `x` é a quantidade da palavra e `n` é q quantidade total.
- Utilize a função `round(valor, casas)` para arredondar a porcentagem para duas casas decimais.
- Palavras mais frequentes devem ser retornadas primeiro.

In [107]:
def frequencia_palavras(rdd, risco, k):
    splitado = rdd.map(lambda x: x.split("|"))
    filtrado = splitado.filter(lambda x: str(x[1]) == str(risco))
    apenas_dados = filtrado.map(lambda x: x[7].lower().replace(".", "").replace(",", ""))
    # print(filtrado.take(2))
    palavras = apenas_dados.map(lambda x: x.split(" "))
    palavras = palavras.flatMap(lambda x: x)
    n = palavras.count()
    # print(palavras.take(15))
    # print(n)
    palavra_contagem = palavras.map(lambda x: (x, 1))
    palavra_contagem = palavra_contagem.reduceByKey(lambda x, y: x+y)
    palavra_contagem = palavra_contagem.sortBy(lambda x: x[1], ascending=False)
    print(palavra_contagem.take(10))
    final = palavra_contagem.map(lambda x: (x[0], round(x[1]/n*100, 2)))
    print(final.take(k))
    # print("------------->")
    return final.take(k)

rdd = sc.textFile("data/log_artico1.txt")

print(frequencia_palavras(rdd, "leve", 1))
print(frequencia_palavras(rdd, "moderado", 3))

[('de', 7), ('observacao', 2), ('formacoes', 2), ('gelo', 2), ('vida', 1), ('na', 1), ('regiao', 1), ('noroeste', 1), ('estudo', 1), ('das', 1)]
[('de', 23.33)]
[('de', 23.33)]
[('de', 27), ('gelo', 8), ('analise', 6), ('da', 4), ('para', 3), ('deslocamento', 3), ('risco', 3), ('abaixo', 2), ('minerais', 2), ('neve', 2)]
[('de', 24.55), ('gelo', 7.27), ('analise', 5.45)]
[('de', 24.55), ('gelo', 7.27), ('analise', 5.45)]


Alguns testes locais:

In [108]:
x = 4.7899
round(x, 2)

4.79

In [109]:
rdd = sc.textFile("data/log_artico2.txt")

assert frequencia_palavras(rdd, "leve", 1) == [('de', 15.79)]
assert frequencia_palavras(rdd, "severo", 5) == [('glacial', 12.0), ('risco', 10.0), ('gelo', 8.0), ('de', 6.0), ('em', 4.0)]

[('de', 3), ('glacial', 2), ('vida', 1), ('na', 1), ('regiao', 1), ('noroeste', 1), ('estudo', 1), ('das', 1), ('amostras', 1), ('observacao', 1)]
[('de', 15.79)]
[('glacial', 6), ('risco', 5), ('gelo', 4), ('de', 3), ('em', 2), ('a', 2), ('ventos', 1), ('visibilidade', 1), ('rachaduras', 1), ('no', 1)]
[('glacial', 12.0), ('risco', 10.0), ('gelo', 8.0), ('de', 6.0), ('em', 4.0)]


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [110]:
ia.sender(answer="frequencia_palavras", task="af_md_24_2", question="ex01c", answer_type="pycode")

interactive(children=(Button(description='Enviar ex01c', style=ButtonStyle()), Output()), _dom_classes=('widge…

Agora que você terminou a parte de programação funcional da prova, salve o notebook e continue a fazer a prova no VS Code.

**Incidentes climáticos**:

<img src="img/afp-20240928-36hc4mv-v2-highres-usweatherhurricanehelene.avif">

O monitoramento climático é crucial para antecipar eventos extremos, mitigar impactos e proteger comunidades vulneráveis. 

Com isto em mente, a base de dados `monitoramento_climatico` foi desenvolvida para gerenciar informações relacionadas a incidentes climáticos ao redor do mundo. Sua estrutura busca capturar aspectos essenciais, desde a localização e severidade até os impactos causados e as medidas tomadas.

A base de dados é composta pelas seguintes tabelas principais:

- **Paises**: Registra os países onde os incidentes são monitorados, permitindo a organização geográfica dos dados.
- **Locais**: Detalha os locais específicos dos incidentes, incluindo latitude, longitude e associação a um país.
- **Severidades**: Define os níveis de severidade de cada incidente, categorizando-os para facilitar a análise de riscos.
- **Tipos de Incidentes**: Especifica os diferentes tipos de incidentes climáticos, como inundações, incêndios florestais, ou tempestades, com descrições que ajudam na identificação e categorização.
- **Incidentes**: Armazena os eventos propriamente ditos, incluindo sua localização, tipo, severidade, data de ocorrência e uma descrição detalhada.
- **Impactos**: Documenta os efeitos de cada incidente, como custos estimados e número de pessoas afetadas, permitindo avaliar a magnitude do evento.
- **Relatórios**: Contém registros textuais detalhados sobre os incidentes, com informações sobre o autor e a data de criação do relatório.

E este é o DER da base:

<img src="img/der_monitoramento_climatico.png">

O script para criação de uma base exemplo está disponível no caminho `./sql/monitoramento_climatico.sql`.

Vamos criar nosso HELPER de conexão com o banco!

In [34]:
import os
import mysql.connector
import insperautograder.jupyter as ia
from functools import partial
from pyspark.sql import SparkSession
from dotenv import load_dotenv

load_dotenv(override=True)

def get_connection_helper(database):

    def run_db_query(connection, query, args=None):
        with connection.cursor() as cursor:
            # print("Executando queries:")
            results = cursor.execute(query, args, multi=True)
            for i, result in enumerate(results):
                if result.with_rows:
                    print(f"Resultado query {i}:")
                    for line in result.fetchall():
                        print(line)
                else:
                    print(f"Query {i} executada!")

    connection = mysql.connector.connect(
        host=os.getenv("MD_DB_SERVER"),
        user=os.getenv("MD_DB_USERNAME"),
        password=os.getenv("MD_DB_PASSWORD"),
        database=database,
    )
    return connection, partial(run_db_query, connection)


connection, db = get_connection_helper("monitoramento_climatico")

Agora responda os exercícios 2 e 3!

**Exercício 2 - Monitoramento climático**:

Considerando que a tabela de incidente possui uma coluna que armazena o custo estimado total dos impactos do incidente (execute no WorkBench):

```sql
ALTER TABLE incidentes
ADD COLUMN custo_estimado_total DECIMAL(15,2) DEFAULT 0.00;
```

Crie uma trigger `trg_insert_impacto` que, após a inserção de um impacto, atualize o `custo_total_impactos` do incidente.

In [36]:
sql_ex02 = """

"""

db(sql_ex02)

Query 0 executada!
Query 1 executada!


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [37]:
ia.sender(answer="sql_ex02", task="af_md_24_2", question="ex02", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex02', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 3 - Custo de incidentes climáticos**:

Crie uma função `custo_total_por_pais` no MySQL. Ela deve receber o nome de um país e retornar o custo total dos impactos que ocorreram no país.

Utilize o mesmo tipo dos custos originais.

In [125]:
sql_ex03 = """
CREATE FUNCTION custo_total_por_pais(P VARCHAR (45)) RETURNS DECIMAL (30,2) READS SQL DATA
BEGIN
    DECLARE S DECIMAL (15,2);
    SELECT IFNULL(SUM(custo_estimado),0) INTO S FROM impactos imp
        LEFT JOIN incidentes inc on inc.id_incidente = imp.id_incidente
        LEFT JOIN locais l on inc.id_local = l.id_local
        LEFT JOIN paises p on p.id_pais = l.id_pais
        WHERE nome_pais LIKE P;
        RETURN S;
END        
"""

db(sql_ex03)

Query 0 executada!


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [126]:
ia.sender(answer="sql_ex03", task="af_md_24_2", question="ex03", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex03', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 4 - Armazenar os dados de monitoramento!**:

<img src="img/Arctic-Report-Card-2021_surface-temperatures_map_graph_2000px.png">

Os dados de monitoramento do ártico precisam ser armazenados, mas precisamos saber quantos discos serão necessários para armazenar todos os arquivos.

Neste exercício, você deve criar uma função para auxiliar no cálculo de quanto custará a compra de discos rígidos para armazenamento dos dados.

**Obs**:
- Considere que os discos serão comprados apenas uma vez, para armazenar uma certa quantidade de arquivos, sendo previamente conhecido o tamanho de cada arquivo.
- Faça os cálculos com ponto flutuante, mas no retorno utilize `math.ceil(x)` para retornar um inteiro arredondado para cima.
- Lembre-se, não é possível comprar meio HD!
- O servidor já terá `import math`.
- Considere o sistema internacional de unidades, ou seja, 1TB = 1000GB e 1GB=1000MB.

Construa uma função `calc_custo_compra_hd` em Python que recebe, nesta ordem:

- `tamanho_arquivo`: tamanho de cada arquivo a ser armazenado, em GB.
- `qtde_arquivos`: quantidade de arquivos.
- `tamanho_hd`: tamanho de cada disco a ser utilizado, em TB.
- `custo_hd`: preço de cada disco.

A função deve calcular e devolver o **custo total** para armazenar os dados no .

In [114]:
import math

def calc_custo_compra_hd(tamanho_arquivo, qtde_arquivos, tamanho_hd, custo_hd):
    qtd_a_ser_armazenado = tamanho_arquivo*qtde_arquivos
    qtd_discos = math.ceil(qtd_a_ser_armazenado/(tamanho_hd*1000))
    return qtd_discos*custo_hd

In [115]:
# Exemplos
print(calc_custo_compra_hd(10, 100, 1, 500))
print(calc_custo_compra_hd(2300, 100, 5, 800))

500
36800


In [116]:
# Alguns testes locais
assert calc_custo_compra_hd(10, 100, 1, 500) == 500
assert calc_custo_compra_hd(10, 100, 5, 500) == 500
assert calc_custo_compra_hd(200, 100, 5, 800) == 3200
assert calc_custo_compra_hd(200, 120, 5, 800) == 4000

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [117]:
ia.sender(answer="calc_custo_compra_hd", task="af_md_24_2", question="ex04", answer_type="pycode")

interactive(children=(Button(description='Enviar ex04', style=ButtonStyle()), Output()), _dom_classes=('widget…

<div class="alert alert-success">

Sua resposta AQUI!

</div>

**Exercício 5 - Futebol no gelo**: (**Nota: 2,5**)



<img src="img/futebol_neve.jpg">

Você foi contratado como engenheiro de dados para organizar informações sobre um campeonato de futebol no gelo.

Atualmente, os dados sobre os jogos, times e jogadores estão armazenados em uma única tabela denormalizada conforme mostrado abaixo:

| id_jogo (PK) | data_jogo   | local_jogo     | id_time (PK) | nome_time     | id_jogador (PK) | nome_jogador   | posição   | gols   | assistências | cartão_amarelo | comissão_tecnica               |
|--------------|-------------|----------------|--------------|---------------|-----------------|----------------|-----------|--------|--------------|----------------|--------------------------------|
| 1            | 2024-11-01 | Arena Glacial  | 10           | Pinguins      | 1001            | Jack Frost     | Goleiro   | 0      | 0            | 1              | ['Jose Antunes', 'Maria Clara'] |
| 1            | 2024-11-01 | Arena Glacial  | 10           | Pinguins      | 1002            | Elsa Snow      | Atacante  | 2      | 1            | 0              | ['Eduardo Guedes', 'John Snow'] |
| 1            | 2024-11-01 | Arena Glacial  | 11           | Ursos Polares | 1003            | Polar Pete     | Defensor  | 0      | 2            | 1              | ['Aluk Sabah']                 |
| 2            | 2024-11-02 | Caverna Gelada | 10           | Pinguins      | 1001            | Jack Frost     | Goleiro   | 0      | 0            | 0              | ['Ter Hah', 'Adams Kelvin']     |
| 2            | 2024-11-02 | Caverna Gelada | 12           | Lobos Árticos | 1004            | Arctic Al      | Atacante  | 1      | 0            | 0              | ['Ana Flores']                  |
| 2            | 2024-11-02 | Caverna Gelada | 12           | Lobos Árticos | 1005            | Icey Ivy       | Defensor  | 0      | 1            | 0              | ['Adams Kelvin']                |
| 3            | 2024-11-03 | Ice Dome       | 13           | Tigres do Gelo| 1006            | Frosty Tom     | Goleiro   | 0      | 0            | 1              | ['Chris Smith']                 |
| 3            | 2024-11-03 | Ice Dome       | 13           | Tigres do Gelo| 1007            | Winter Grace   | Atacante  | 2      | 1            | 0              | ['Chris Smith']                 |
| 3            | 2024-11-03 | Ice Dome       | 14           | Focas do Ártico | 1008          | Arctic Joe     | Defensor  | 1      | 0            | 0              | ['Alex Turner']                 |
| 3            | 2024-11-03 | Ice Dome       | 14           | Focas do Ártico | 1009          | Snowy Sue      | Atacante  | 0      | 1            | 1              | ['Alex Turner']                 |
| 4            | 2024-11-04 | Glacier Arena  | 10           | Pinguins      | 1001            | Jack Frost     | Goleiro   | 0      | 0            | 0              | ['Jose Antunes', 'Maria Clara'] |
| 4            | 2024-11-04 | Glacier Arena  | 15           | Lobos do Norte | 1010          | Frosty Frank   | Atacante  | 3      | 2            | 0              | ['Carla Mendes']                |
| 4            | 2024-11-04 | Glacier Arena  | 15           | Lobos do Norte | 1011          | Icy Fiona      | Defensor  | 0      | 1            | 0              | ['Carla Mendes']                |
| 5            | 2024-11-05 | Frost Field    | 14           | Focas do Ártico | 1008          | Arctic Joe     | Defensor  | 0      | 2            | 1              | ['Alex Turner']                 |
| 5            | 2024-11-05 | Frost Field    | 16           | Ursos do Gelo | 1012           | Icey Ingrid    | Atacante  | 2      | 1            | 0              | ['Bruce Winter']                |
| 5            | 2024-11-05 | Frost Field    | 16           | Ursos do Gelo | 1013           | Frozen Fred    | Goleiro   | 0      | 0            | 0              | ['Bruce Winter']                |
| 6            | 2024-11-06 | Ice Dome       | 13           | Tigres do Gelo| 1006            | Frosty Tom     | Goleiro   | 0      | 0            | 0              | ['Chris Smith']                 |
| 6            | 2024-11-06 | Ice Dome       | 15           | Lobos do Norte | 1010          | Frosty Frank   | Atacante  | 2      | 2            | 0              | ['Carla Mendes']                |
| 6            | 2024-11-06 | Ice Dome       | 15           | Lobos do Norte | 1011          | Icy Fiona      | Defensor  | 1      | 1            | 0              | ['Carla Mendes']                |
| 6            | 2024-11-06 | Ice Dome       | 13           | Tigres do Gelo| 1007            | Winter Grace   | Atacante  | 3      | 1            | 1              | ['Chris Smith']                 |

**Descrição dos Campos:**

- `id_jogo (PK)`: Identificador único do jogo.
- `data_jogo`: Data do jogo.
- `local_jogo`: Local onde o jogo ocorreu.
- `id_time (PK)`: Identificador único do time.
- `nome_time`: Nome do time.
- `id_jogador (PK)`: Identificador único do jogador.
- `nome_jogador`: Nome do jogador.
- `posição`: Posição do jogador (Goleiro, Atacante, Defensor, etc.).
- `gols`: Número de gols marcados pelo jogador no jogo.
- `assistências`: Número de assistências realizadas pelo jogador no jogo.
- `cartão_amarelo`: Número de cartões amarelos recebidos pelo jogador.
- `comissão_tecnica`: Lista de técnicos associados ao time.

**Tarefa**:

Baseado na tabela denormalizada fornecida, normalize os dados até a 3ª Forma Normal (3NF), garantindo:

1. Eliminação de redundâncias.
2. Integridade referencial entre as entidades.
3. Relacionamento correto entre jogos, times, jogadores e técnicos.

**Observações**:
- Considere que cada jogo sempre é entre dois times.

Crie no **MySQL Workbench** ou **Draw.io** o **Diagrama de Entidades e Relacionamentos (DER)** que representa a base de dados normalizada.

**Instruções:**
- Salve o DER na pasta `resposta_ex05`. Utilize formato png ou jpg.
- Insira o caminho do arquivo no notebook usando uma tag para exibir a imagem.

**Critérios de Avaliação (Rubrica):**

| Conceito | Nota Porcentual | Descrição                                                                                                                                               |
|:----------:|----------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------|
| I        | 0.0 |Insuficiente, as modificações propostas não consideram as informações a serem armazenadas ou consideram um cenário diferente do proposto |
| D        | 0.3 |Não está na 1NF, mas a solução está em desenvolvimento e permite armazenar boa parte das informações |
| C        | 0.6 |Está na 1NF, sem erros, redundâncias ou ineficiências | |
| B        | 0.8 |Está na 2NF, sem erros, redundâncias ou ineficiências | |
| A        | 1.0 |Está na 3NF, sem erros, redundâncias ou ineficiências |


<div class="alert alert-success">

Seu diagrama do workbench AQUI!

<img src="resposta_ex05/resposta.png">

</div>

<div class="alert alert-success">

1NF:
- A tabela fornecida possui colunas com valores atômicos e cada celula contem apenas 1 valor
- Não há repetição de grupo de campos

Para chegar nesse nivel, removi a comissao tecnica no formato de listas anterior, que nao garante a atomicidade

2NF
- Tabela em 1NF
- Não deve haver dependências parciais
Agora tudo que nao é chave primaria, depende da chave primaria completa. Antigamente, informaçoes como nome do jogador dependia apenas da chave primaria id_jogador, mas nao dependia da chave id_jogo

3NF
- Tabela em 2NF
- Não deve haver dependência transitiva (um atributo não chave não deve depender de outro atributo não chave)
com as tabelas separadas, em cada tabela as informaçoes dependem da chave

</div>

## Referências das imagens

- https://www.ultimadivisao.com.br/wp-content/uploads/2022/04/12081164-429039237304921-1524049854-n-800.jpg
- https://images.stockcake.com/public/b/c/0/bc05dc8b-fd16-4d8f-bc8f-c4d74c83e9ef_large/arctic-scientific-research-stockcake.jpg
- https://s2-oglobo.glbimg.com/bng5jZCwL_uBVq9ZYF6sP60EDx0=/0x0:3900x2600/888x0/smart/filters:strip_icc()/i.s3.glbimg.com/v1/AUTH_da025474c0c44edd99332dddb09cabe8/internal_photos/bs/2024/b/L/kER5XJStOC9DWACN5Jqg/afp-20240928-36hc4mv-v2-highres-usweatherhurricanehelene.jpg
- https://www.climate.gov/sites/default/files/styles/full_width_620_original_image/public/2021-12/Arctic-Report-Card-2021_surface-temperatures_map_graph_2000px.png?itok=Rg6nT8wG

### Conferindo as notas!

In [131]:
ia.grades(task="af_md_24_2")

|    | Atividade   | Exercício   |   Peso |   Nota |
|---:|:------------|:------------|-------:|-------:|
|  0 | af_md_24_2  | ex01a       |      2 |     10 |
|  1 | af_md_24_2  | ex01b       |      4 |     10 |
|  2 | af_md_24_2  | ex01c       |      5 |     10 |
|  3 | af_md_24_2  | ex02        |      2 |      0 |
|  4 | af_md_24_2  | ex03        |      4 |     10 |
|  5 | af_md_24_2  | ex04        |      4 |     10 |

In [132]:
ia.grades(by="TASK", task="af_md_24_2")

|    | Tarefa     |   Nota |
|---:|:-----------|-------:|
|  0 | af_md_24_2 |   9.05 |

## Entrega!

É hora de entregar. Faça **zip** (não RAR), envie no Blackboard e finalize o teste no proctorio!